In [ ]:
import re

file_path = r"c:\Users\Striker\Documents\shopping-intranet\whatsapp-broker\src\controllers\SuperAdminController.js"

with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()

new_method = """
  static async getSystemStats(req, res) {
    try {
      const tenants = await prisma.tenant.findMany({
        include: {
            users: true,
            _count: {
                select: { 
                    conversations: true
                }
            }
        }
      });
      
      const stats = [];
      
      for (const t of tenants) {
          const messageCount = await prisma.message.count({
              where: {
                  conversation: {
                      tenantId: t.id
                  },
                  direction: 'OUTBOUND'
              }
          });

          const activeUsers = t.users.length;
          const totalCost = (messageCount * (t.costPerMessage || 0)) + (activeUsers * (t.costPerUser || 0));

          stats.push({
              tenantId: t.id,
              name: t.name,
              plan: t.plan,
              active: t.active,
              users: activeUsers,
              conversations: t._count.conversations,
              messagesSent: messageCount,
              unitCostMsg: t.costPerMessage || 0,
              unitCostUser: t.costPerUser || 0,
              estimatedCost: totalCost.toFixed(2),
              currency: t.currency || 'BRL'
          });
      }

      const totalTenants = tenants.length;
      const totalRevenue = stats.reduce((acc, curr) => acc + parseFloat(curr.estimatedCost), 0).toFixed(2);
      const totalMessages = stats.reduce((acc, curr) => acc + curr.messagesSent, 0);

      res.json({
          overview: {
              totalTenants,
              totalRevenue,
              totalMessages,
              systemStatus: 'ONLINE'
          },
          details: stats
      });

    } catch (e) {
      console.error(e);
      res.status(500).json({ error: 'Failed to get system stats' });
    }
  }
"""

last_brace_index = content.rfind('}')
if last_brace_index != -1:
    new_content = content[:last_brace_index] + new_method + content[last_brace_index:]
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(new_content)
    print("Method added successfully.")
else:
    print("Could not find closing brace.")


In [ ]:
import re

file_path = r"c:\Users\Striker\Documents\shopping-intranet\whatsapp-broker\src\controllers\SupervisorController.js"

with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()

new_method = """
    static async getHistory(req, res) {
        try {
            const tenantId = req.tenantId;
            const limit = parseInt(req.query.limit) || 50;

            const conversations = await prisma.conversation.findMany({
                where: {
                    tenantId,
                    status: 'RESOLVED'
                },
                include: {
                    contact: true,
                    assignedTo: { select: { name: true, id: true } },
                    messages: {
                        take: 1,
                        orderBy: { createdAt: 'desc' }
                    }
                },
                orderBy: { updatedAt: 'desc' },
                take: limit
            });

            const formatted = conversations.map(c => ({
                id: c.id,
                contactName: c.contact.name || c.contact.waId,
                status: c.status,
                assignedTo: c.assignedTo ? c.assignedTo.name : 'Finalizado',
                lastMessage: c.messages[0]?.content || '',
                waitingSince: c.updatedAt
            }));

            res.json(formatted);
        } catch (e) {
            console.error(e);
            res.status(500).json({ error: 'Erro ao buscar histórico' });
        }
    }
"""

last_brace_index = content.rfind('}')
if last_brace_index != -1:
    new_content = content[:last_brace_index] + new_method + content[last_brace_index:]
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(new_content)
    print("Method added successfully.")
else:
    print("Could not find closing brace.")
